# RizeHire: BERT + Residual ANN on the 100K Resume–Job Dataset (V3)
**Sandeep Kumar (2022UG1113) & Nikhil Sonkar (2022UG1111)**
**IIIT Ranchi | Major Project | 2026**

Same hybrid pipeline as V2, but trained on the **100,000-row `dataset.csv`** (columns: `ID, Name, Role, Transcript, Resume, decision, Reason_for_decision, Job_Description`) that the cross-encoder in `RizeHire_BERT_FineTuning.ipynb` was fine-tuned on. Every row has its **own** job description, so all BERT features are computed per resume–job pair.

**Pipeline:** Resume + JD → Sentence-BERT embeddings (384-d each) → interaction features (product, |diff|) → Cross-Encoder pair score + cosine → structured features → StandardScaler → **Residual ANN** (512 → ResBlock×2 → 256 → 128 → 2) → Select / Reject

**Outputs:** `bert_ann_100k_metrics.json` (the numbers for your resume), `rizehire_bert_ann_100k.pth`, and the PNG plots.

### How to use
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all. Upload `dataset.csv` when prompted (or set `DATASET_PATH` to a Drive path).
3. Expected time on a T4: ~5 min SBERT encoding + ~10–15 min cross-encoder scoring + ~5 min ANN training.


## Step 1: Setup

In [ ]:
!pip install -q sentence-transformers torch scikit-learn matplotlib seaborn pandas numpy

import os, json, time, math
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4)')


## Step 2: Configuration

- `USE_TRANSCRIPT=True` appends interview highlights to the resume, matching the cross-encoder fine-tuning setup (hire decisions in this dataset depend on the interview). Set to `False` for a pure resume–JD experiment.
- `MAX_SAMPLES=None` uses all 100K rows. Set e.g. `20000` for a quick dry run.
- `CROSS_ENCODER_PATH`: if you upload/unzip `rizehire_cross_encoder_final/` here, the fine-tuned model is used for the pair score; otherwise the base `ms-marco-MiniLM-L-6-v2`.


In [ ]:
DATASET_PATH = None            # e.g. '/content/drive/MyDrive/dataset.csv'; None = upload prompt
USE_TRANSCRIPT = True
MAX_SAMPLES = None             # None = all rows
RANDOM_SEED = 42
CROSS_ENCODER_PATH = '/content/rizehire_cross_encoder_final'   # fine-tuned dir if present
BASE_CROSS_ENCODER = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
SBERT_MODEL = 'all-MiniLM-L6-v2'

RESUME_CHARS, TRANSCRIPT_CHARS, JD_CHARS = 1500, 1000, 500
EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, PATIENCE = 40, 256, 5e-4, 1e-3, 10
RUN_SHAP = False               # KernelSHAP on 1,5xx features is slow; enable for the report figure

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)


## Step 3: Load dataset

In [ ]:
if DATASET_PATH is None:
    from google.colab import files
    print('Upload dataset.csv:')
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATASET_PATH)
print('Loaded {} rows, columns: {}'.format(len(df), list(df.columns)))

# Tolerate column-name variants (same logic as train_bert_model.py)
def find_col(candidates):
    for c in df.columns:
        if c.lower() in candidates:
            return c
    return None

resume_col = find_col(['resume', 'resume_text', 'cv'])
jd_col = find_col(['job_description', 'jd', 'job_desc', 'description'])
label_col = find_col(['decision', 'label', 'result', 'outcome'])
transcript_col = find_col(['transcript', 'interview', 'interview_transcript'])
assert resume_col and jd_col and label_col, 'Could not find Resume / Job_Description / decision columns'
print('Using columns: resume={!r} jd={!r} label={!r} transcript={!r}'.format(resume_col, jd_col, label_col, transcript_col))

label_map = {'select': 1, 'hire': 1, 'hired': 1, 'yes': 1, 'accept': 1, 'selected': 1, '1': 1,
             'reject': 0, 'rejected': 0, 'no': 0, 'decline': 0, '0': 0}
df['label'] = df[label_col].astype(str).str.lower().str.strip().map(label_map)
df = df.dropna(subset=['label', resume_col, jd_col]).copy()
df['label'] = df['label'].astype(int)

if MAX_SAMPLES and len(df) > MAX_SAMPLES:
    df, _ = train_test_split(df, train_size=MAX_SAMPLES, random_state=RANDOM_SEED, stratify=df['label'])
    df = df.reset_index(drop=True)

print('After cleaning: {} rows | label distribution: {}'.format(len(df), df['label'].value_counts().to_dict()))


## Step 4: Build text pairs

In [ ]:
resume_txt = df[resume_col].astype(str).str[:RESUME_CHARS]
if USE_TRANSCRIPT and transcript_col:
    df['candidate_text'] = resume_txt + ' [SEP] Interview highlights: ' + df[transcript_col].astype(str).str[:TRANSCRIPT_CHARS]
    print('Candidate text = resume + interview transcript')
else:
    df['candidate_text'] = resume_txt
    print('Candidate text = resume only')
df['jd_text'] = df[jd_col].astype(str).str[:JD_CHARS]

candidate_texts = df['candidate_text'].tolist()
jd_texts = df['jd_text'].tolist()
labels = df['label'].values
print('Avg candidate chars: {:.0f} | avg JD chars: {:.0f}'.format(
    df['candidate_text'].str.len().mean(), df['jd_text'].str.len().mean()))


## Step 5: BERT features (SBERT embeddings + Cross-Encoder pair score)

In [ ]:
t0 = time.time()
sbert = SentenceTransformer(SBERT_MODEL, device=device)
print('SBERT loaded ({}-dim)'.format(sbert.get_sentence_embedding_dimension()))

print('Encoding candidate texts...')
cand_emb = sbert.encode(candidate_texts, batch_size=256, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)
print('Encoding job descriptions...')
jd_emb = sbert.encode(jd_texts, batch_size=256, show_progress_bar=True,
                      convert_to_numpy=True, normalize_embeddings=True)
print('Embeddings: {} / {}  ({:.0f}s)'.format(cand_emb.shape, jd_emb.shape, time.time() - t0))


In [ ]:
t0 = time.time()
if os.path.isdir(CROSS_ENCODER_PATH):
    cross_enc = CrossEncoder(CROSS_ENCODER_PATH, max_length=512, device=device)
    cross_encoder_used = 'fine-tuned: ' + CROSS_ENCODER_PATH
else:
    cross_enc = CrossEncoder(BASE_CROSS_ENCODER, max_length=512, device=device)
    cross_encoder_used = BASE_CROSS_ENCODER
print('Cross-encoder:', cross_encoder_used)

pairs = list(zip(candidate_texts, jd_texts))
cross_scores = cross_enc.predict(pairs, batch_size=128, show_progress_bar=True)
cross_scores = np.asarray(cross_scores, dtype=np.float32).reshape(-1, 1)
print('Cross-encoder scores: {} range [{:.3f}, {:.3f}] ({:.0f}s)'.format(
    cross_scores.shape, cross_scores.min(), cross_scores.max(), time.time() - t0))
print('  mean score | reject: {:.3f} | select: {:.3f}'.format(
    cross_scores[labels == 0].mean(), cross_scores[labels == 1].mean()))


## Step 6: Feature fusion

In [ ]:
cosine_sims = np.sum(cand_emb * jd_emb, axis=1, keepdims=True).astype(np.float32)   # embeddings are L2-normalised
elem_product = (cand_emb * jd_emb).astype(np.float32)
elem_diff = np.abs(cand_emb - jd_emb).astype(np.float32)

structured = np.column_stack([
    df['candidate_text'].str.len().values / 100.0,
    df['jd_text'].str.len().values / 100.0,
    df['candidate_text'].str.split().str.len().values / 10.0,
    df['jd_text'].str.split().str.len().values / 10.0,
]).astype(np.float32)

X = np.concatenate([cand_emb, jd_emb, elem_product, elem_diff, cross_scores, cosine_sims, structured], axis=1)
feature_blocks = {
    'SBERT candidate (384)': (0, 384), 'SBERT job (384)': (384, 768),
    'candidate*job (384)': (768, 1152), '|candidate-job| (384)': (1152, 1536),
    'cross-encoder score (1)': (1536, 1537), 'cosine similarity (1)': (1537, 1538),
    'structured (4)': (1538, X.shape[1]),
}
print('FINAL FEATURE VECTOR: {}'.format(X.shape))
for k, (a, b) in feature_blocks.items():
    print('  {:<28} dims {}-{}'.format(k, a, b))


## Step 7: Scale, split (80/10/10, stratified), class weights

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)
num_classes = 2

X_trval, X_test, y_trval, y_test = train_test_split(X_scaled, labels, test_size=0.1, random_state=RANDOM_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.111, random_state=RANDOM_SEED, stratify=y_trval)
print('Train {} | Val {} | Test {}'.format(len(X_train), len(X_val), len(X_test)))

class_counts = Counter(y_train)
class_weights = {c: len(y_train) / (num_classes * n) for c, n in class_counts.items()}
weight_tensor = torch.FloatTensor([class_weights[i] for i in range(num_classes)]).to(device)
print('Class counts:', dict(class_counts), '| weights:', {k: round(v, 3) for k, v in class_weights.items()})

sample_weights = [class_weights[y] for y in y_train]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long()),
                          batch_size=BATCH_SIZE, sampler=sampler, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).long()),
                        batch_size=1024, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).long()),
                         batch_size=1024, shuffle=False)


## Step 8: Residual ANN (same architecture as V2)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.activation(x + self.block(x)))


class ResumeClassifierANN(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.3))
        self.res1 = ResidualBlock(512, dropout=0.3)
        self.res2 = ResidualBlock(512, dropout=0.25)
        self.reduce = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.15),
        )
        self.classifier = nn.Linear(128, num_classes)
    def forward(self, x):
        return self.classifier(self.reduce(self.res2(self.res1(self.input_proj(x)))))


input_dim = X_scaled.shape[1]
model = ResumeClassifierANN(input_dim, num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print('Architecture: {} -> 512 -> ResBlock x2 -> 256 -> 128 -> {}'.format(input_dim, num_classes))
print('Parameters: {:,}'.format(total_params))


## Step 9: Train (AdamW + cosine warm restarts, class-weighted CE + label smoothing, early stopping on val F1)

In [ ]:
def evaluate(loader):
    model.eval(); preds, trues, probs, loss_sum = [], [], [], 0.0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            out = model(bx)
            loss_sum += criterion(out, by).item() * len(by)
            p = torch.softmax(out, dim=1)[:, 1]
            preds.extend(out.argmax(1).cpu().numpy()); trues.extend(by.cpu().numpy()); probs.extend(p.cpu().numpy())
    preds, trues, probs = np.array(preds), np.array(trues), np.array(probs)
    return {
        'loss': loss_sum / len(trues),
        'acc': accuracy_score(trues, preds),
        'f1': f1_score(trues, preds),                       # binary F1, positive class = select
        'f1_weighted': f1_score(trues, preds, average='weighted'),
        'precision': precision_score(trues, preds, zero_division=0),
        'recall': recall_score(trues, preds, zero_division=0),
        'preds': preds, 'trues': trues, 'probs': probs,
    }

criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}
best_f1, best_epoch, patience_counter = 0.0, 0, 0
t0 = time.time()

for epoch in range(EPOCHS):
    model.train(); loss_sum, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(bx); loss = criterion(out, by); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item() * len(by); correct += (out.argmax(1) == by).sum().item(); total += len(by)
    scheduler.step(epoch)

    val = evaluate(val_loader)
    history['train_loss'].append(loss_sum / total); history['train_acc'].append(correct / total)
    history['val_loss'].append(val['loss']); history['val_acc'].append(val['acc']); history['val_f1'].append(val['f1'])
    history['lr'].append(optimizer.param_groups[0]['lr'])

    marker = ''
    if val['f1'] > best_f1:
        best_f1, best_epoch, patience_counter = val['f1'], epoch + 1, 0
        torch.save(model.state_dict(), 'best_model.pth'); marker = '  *** best'
    else:
        patience_counter += 1
    print('Epoch {:3d}/{} | train acc {:.4f} loss {:.4f} | val acc {:.4f} f1 {:.4f} loss {:.4f} | lr {:.2e}{}'.format(
        epoch + 1, EPOCHS, correct / total, loss_sum / total, val['acc'], val['f1'], val['loss'], history['lr'][-1], marker))
    if patience_counter >= PATIENCE:
        print('Early stopping (no val-F1 improvement for {} epochs)'.format(PATIENCE)); break

train_seconds = time.time() - t0
print('\nBest val F1 {:.4f} at epoch {} ({:.0f}s)'.format(best_f1, best_epoch, train_seconds))


## Step 10: Test-set evaluation

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
test = evaluate(test_loader)
fpr, tpr, _ = roc_curve(test['trues'], test['probs']); roc_auc = auc(fpr, tpr)
cm = confusion_matrix(test['trues'], test['preds'])

print('=' * 60)
print('   BERT + RESIDUAL ANN — TEST RESULTS ({} rows)'.format(len(test['trues'])))
print('=' * 60)
print('   Accuracy:        {:.2f}%'.format(test['acc'] * 100))
print('   Precision:       {:.2f}%'.format(test['precision'] * 100))
print('   Recall:          {:.2f}%'.format(test['recall'] * 100))
print('   F1 (select):     {:.4f}'.format(test['f1']))
print('   F1 (weighted):   {:.4f}'.format(test['f1_weighted']))
print('   ROC-AUC:         {:.4f}'.format(roc_auc))
print('=' * 60)
print(classification_report(test['trues'], test['preds'], target_names=['Reject', 'Select'], digits=4))
print('Confusion matrix [[TN FP] [FN TP]]:\n', cm)


## Step 11: Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history['train_acc']) + 1)
axes[0].plot(ep, [a * 100 for a in history['train_acc']], label='Train'); axes[0].plot(ep, [a * 100 for a in history['val_acc']], label='Val')
axes[0].set_title('Accuracy (%)'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(ep, history['train_loss'], label='Train'); axes[1].plot(ep, history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=.3)
axes[2].plot(ep, history['val_f1'], color='purple'); axes[2].set_title('Val F1'); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('bert_ann_100k_training_curves.png', dpi=200); plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, xticklabels=['Reject', 'Select'], yticklabels=['Reject', 'Select'])
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual'); ax1.set_title('Confusion Matrix (acc {:.2f}%)'.format(test['acc'] * 100))
ax2.plot(fpr, tpr, lw=2, label='BERT+ResANN (AUC = {:.3f})'.format(roc_auc)); ax2.plot([0, 1], [0, 1], 'r--', lw=1)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.set_title('ROC'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.savefig('bert_ann_100k_evaluation.png', dpi=200); plt.show()


## Step 12 (optional): SHAP feature-group importance

In [ ]:
if RUN_SHAP:
    !pip install -q shap
    import shap
    def predict_fn(Xn):
        model.eval()
        with torch.no_grad():
            return torch.softmax(model(torch.FloatTensor(Xn).to(device)), dim=1).cpu().numpy()
    explainer = shap.KernelExplainer(predict_fn, X_train[:30])
    sv = explainer.shap_values(X_test[:10])
    sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if sv.ndim == 3 else sv)
    sv_abs = np.abs(sv)
    groups = {k: float(sv_abs[:, a:b].mean()) for k, (a, b) in feature_blocks.items()}
    plt.figure(figsize=(10, 5)); plt.barh(list(groups.keys()), list(groups.values()))
    plt.xlabel('Mean |SHAP|'); plt.title('Feature-group importance'); plt.tight_layout()
    plt.savefig('bert_ann_100k_shap_groups.png', dpi=200); plt.show()
    print(groups)
else:
    groups = None
    print('RUN_SHAP=False, skipped')


## Step 13: Save metrics JSON + model, download

`bert_ann_100k_metrics.json` is the file to keep. Put **these** numbers on the resume:
`accuracy` and `f1_select` (or `f1_weighted`, say which).


In [ ]:
metrics = {
    'experiment': 'BERT + Residual ANN (V3, 100K dataset)',
    'trained_at': datetime.now().isoformat(),
    'dataset': os.path.basename(DATASET_PATH),
    'total_samples': int(len(df)),
    'train_samples': int(len(X_train)), 'val_samples': int(len(X_val)), 'test_samples': int(len(X_test)),
    'use_transcript': bool(USE_TRANSCRIPT and transcript_col is not None),
    'sbert_model': SBERT_MODEL,
    'cross_encoder': cross_encoder_used,
    'feature_dim': int(input_dim),
    'feature_blocks': {k: list(v) for k, v in feature_blocks.items()},
    'architecture': '{} -> 512 -> ResBlock x2 -> 256 -> 128 -> 2 (GELU, BatchNorm, Dropout, skip connections)'.format(input_dim),
    'total_parameters': int(total_params),
    'hyperparameters': {'epochs_max': EPOCHS, 'epochs_run': len(history['val_f1']), 'best_epoch': best_epoch,
                        'batch_size': BATCH_SIZE, 'lr': LR, 'weight_decay': WEIGHT_DECAY, 'patience': PATIENCE,
                        'optimizer': 'AdamW', 'scheduler': 'CosineAnnealingWarmRestarts(T_0=10,T_mult=2)',
                        'loss': 'CrossEntropy + class weights + label_smoothing=0.05', 'grad_clip': 1.0,
                        'train_seconds': round(train_seconds, 1)},
    'validation': {'best_f1_select': round(float(best_f1), 4), 'best_acc': round(float(history['val_acc'][best_epoch - 1]), 4)},
    'test': {
        'accuracy': round(float(test['acc']), 4),
        'precision_select': round(float(test['precision']), 4),
        'recall_select': round(float(test['recall']), 4),
        'f1_select': round(float(test['f1']), 4),
        'f1_weighted': round(float(test['f1_weighted']), 4),
        'roc_auc': round(float(roc_auc), 4),
        'confusion_matrix': cm.tolist(),
    },
    'shap_group_importance': groups,
    'platform': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
}
with open('bert_ann_100k_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics['test'], indent=2))

torch.save({'model_state_dict': model.state_dict(), 'input_dim': input_dim, 'num_classes': num_classes,
            'scaler_mean': scaler.mean_.tolist(), 'scaler_scale': scaler.scale_.tolist(), 'metrics': metrics},
           'rizehire_bert_ann_100k.pth')
print('Saved rizehire_bert_ann_100k.pth and bert_ann_100k_metrics.json')


In [ ]:
from google.colab import files
for f in ['bert_ann_100k_metrics.json', 'rizehire_bert_ann_100k.pth',
          'bert_ann_100k_training_curves.png', 'bert_ann_100k_evaluation.png', 'bert_ann_100k_shap_groups.png']:
    if os.path.exists(f):
        try:
            files.download(f); print('Downloaded', f)
        except Exception as e:
            print('Download manually:', f, e)
print('\nCopy bert_ann_100k_metrics.json into python-explainability/ so the numbers live in the repo.')
